<center>
	Пример 1 с сложным @Redirect
</center>

Код метода внутри класса `TitleScreen` :

```java
private void initWidgetsNormal(int y, int spacingY) {
	this.addDrawableChild(
		ButtonWidget.builder(Text.translatable("menu.singleplayer"), button -> this.client.setScreen(new SelectWorldScreen(this)))
			.dimensions(this.width / 2 - 100, y, 200, 20)
			.build()
	);
	Text text = this.getMultiplayerDisabledText();
	boolean bl = text == null;
	Tooltip tooltip = text != null ? Tooltip.of(text) : null;
	this.addDrawableChild(ButtonWidget.builder(Text.translatable("menu.multiplayer"), button -> {
		Screen screen = (Screen)(this.client.options.skipMultiplayerWarning ? new MultiplayerScreen(this) : new MultiplayerWarningScreen(this));
		this.client.setScreen(screen);
	}).dimensions(this.width / 2 - 100, y + spacingY * 1, 200, 20).tooltip(tooltip).build()).active = bl;
	this.addDrawableChild(
			ButtonWidget.builder(Text.translatable("menu.online"), button -> this.switchToRealms())
				.dimensions(this.width / 2 - 100, y + spacingY * 2, 200, 20)
				.tooltip(tooltip)
				.build()
		)
		.active = bl;
}
```

Код внутри класса mixins для `TitleScreen` :

```java
@Redirect(method = "initWidgetsNormal", 
              at = @At(value = "INVOKE", 
                       target = "Lnet/minecraft/client/gui/screen/TitleScreen;addDrawableChild(Lnet/minecraft/client/gui/Element;)Lnet/minecraft/client/gui/Element;",
					   ordinal = 2))
private Element redirectAddDrawableChild(TitleScreen instance, Element drawableElement) {
        return this.addDrawableChild(
            ButtonWidget.builder(Text.translatable("menu.mods"), btn -> System.out.println("Mods clicked"))
                .dimensions(((ButtonWidget)drawableElement).getX(), ((ButtonWidget)drawableElement).getY(), ((ButtonWidget)drawableElement).getWidth(), ((ButtonWidget)drawableElement).getHeight())
                    .build()
        );
}
```

Вся сложность заключается в правильном понимании байткода `target`, так как целевой метод является дженериком реализующим интерфейс, кроме того подтягивается с родительского класса.

```java
protected <T extends Element & Drawable & Selectable> T addDrawableChild(T drawableElement) {
	this.drawables.add(drawableElement);
	return this.addSelectableChild(drawableElement);
}
```

Определение находится внутри класса `Screen`, но объект вызывается из класса `TitleScreen` с помощью ключевого слова `this`, поэтому вызывается не объект принадлежащий `Screen`, а его копия в `TitleScreen`, подхваченная наследованием, засчёт модификатора доступа `protected`, поэтому в `target` указывается `Lnet/minecraft/client/gui/screen/TitleScreen;`.

В свою очередь для определения типа дженерика нужно обратиться к байткоду : 

```java
private void a(int arg0, int arg1) { //(II)V
    <localVar:index=0 , name=this , desc=Leuw;, sig=null, start=L0, end=L13>
    <localVar:index=1 , name=$$0 , desc=I, sig=null, start=L0, end=L13>
    <localVar:index=2 , name=$$1 , desc=I, sig=null, start=L0, end=L13>
    <localVar:index=3 , name=$$2 , desc=Lsw;, sig=null, start=L2, end=L13>
    <localVar:index=4 , name=$$3 , desc=Z, sig=null, start=L5, end=L13>
    <localVar:index=5 , name=$$4 , desc=Leqp;, sig=null, start=L8, end=L13

    L0 {
        aload 0 // reference to self
        ldc "menu.singleplayer" (java.lang.String)
        invokestatic sw.c(Ljava/lang/String;)Ltj;
        aload 0 // reference to self
        invokedynamic java/lang/invoke/LambdaMetafactory.metafactory(Ljava/lang/invoke/MethodHandles$Lookup;Ljava/lang/String;Ljava/lang/invoke/MethodType;Ljava/lang/invoke/MethodType;Ljava/lang/invoke/MethodHandle;Ljava/lang/invoke/MethodType;)Ljava/lang/invoke/CallSite; : onPress(Leuw;)Lepi$c; (Lepi;)V euw.d(Lepi;)V (5) (Lepi;)V
        invokestatic epi.a(Lsw;Lepi$c;)Lepi$a;
        aload 0 // reference to self
        getfield euw.g:int
        iconst_2
        idiv
        bipush 100
        isub
        iload 1 // reference to arg0
        sipush 200
        bipush 20
        invokevirtual epi$a.a(IIII)Lepi$a;
        invokevirtual epi$a.a()Lepi;
        invokevirtual euw.d(Leqt;)Leqt;
        pop
    }
    L1 {
        aload 0 // reference to self
        invokevirtual euw.B()Lsw;
        astore 3
    }
    L2 {
        aload 3
        ifnonnull L3
        iconst_1
        goto L4
    }
    L3 {
        f_new (Locals[4]: euw, int, int, sw) (Stack[0]) 
        iconst_0
    }
    L4 {
        f_new (Locals[4]: euw, int, int, sw) (Stack[1]: int) 
        istore 4
    }
    L5 {
        aload 3
        ifnull L6
        aload 3
        invokestatic eqp.a(Lsw;)Leqp;
        goto L7
    }
    L6 {
        f_new (Locals[5]: euw, int, int, top, int) (Stack[0]) 
        aconst_null
    }
    L7 {
        f_new (Locals[5]: euw, int, int, top, int) (Stack[1]: eqp) 
        astore 5
    }
    L8 {
        aload 0 // reference to self
        ldc "menu.multiplayer" (java.lang.String)
        invokestatic sw.c(Ljava/lang/String;)Ltj;
        aload 0 // reference to self
        invokedynamic java/lang/invoke/LambdaMetafactory.metafactory(Ljava/lang/invoke/MethodHandles$Lookup;Ljava/lang/String;Ljava/lang/invoke/MethodType;Ljava/lang/invoke/MethodType;Ljava/lang/invoke/MethodHandle;Ljava/lang/invoke/MethodType;)Ljava/lang/invoke/CallSite; : onPress(Leuw;)Lepi$c; (Lepi;)V euw.c(Lepi;)V (5) (Lepi;)V
        invokestatic epi.a(Lsw;Lepi$c;)Lepi$a;
        aload 0 // reference to self
        getfield euw.g:int
        iconst_2
        idiv
        bipush 100
        isub
        iload 1 // reference to arg0
        iload 2
        iconst_1
        imul
        iadd
        sipush 200
        bipush 20
    }
    L9 {
        invokevirtual epi$a.a(IIII)Lepi$a;
        aload 5
        invokevirtual epi$a.a(Leqp;)Lepi$a;
        invokevirtual epi$a.a()Lepi;
    }
    L10 {
        invokevirtual euw.d(Leqt;)Leqt; // Нужная INVOKE инструкция
        checkcast epi
        iload 4
        putfield epi.r:boolean
    }
    L11 {
        aload 0 // reference to self
        ldc "menu.online" (java.lang.String)
        invokestatic sw.c(Ljava/lang/String;)Ltj;
        aload 0 // reference to self
        invokedynamic java/lang/invoke/LambdaMetafactory.metafactory(Ljava/lang/invoke/MethodHandles$Lookup;Ljava/lang/String;Ljava/lang/invoke/MethodType;Ljava/lang/invoke/MethodType;Ljava/lang/invoke/MethodHandle;Ljava/lang/invoke/MethodType;)Ljava/lang/invoke/CallSite; : onPress(Leuw;)Lepi$c; (Lepi;)V euw.b(Lepi;)V (5) (Lepi;)V
        invokestatic epi.a(Lsw;Lepi$c;)Lepi$a;
        aload 0 // reference to self
        getfield euw.g:int
        iconst_2
        idiv
        bipush 100
        isub
        iload 1 // reference to arg0
        iload 2
        iconst_2
        imul
        iadd
        sipush 200
        bipush 20
        invokevirtual epi$a.a(IIII)Lepi$a;
        aload 5
        invokevirtual epi$a.a(Leqp;)Lepi$a;
        invokevirtual epi$a.a()Lepi;
        invokevirtual euw.d(Leqt;)Leqt;
        checkcast epi
        iload 4
        putfield epi.r:boolean
    }
    L12 {
        return
    }
    L13 {
    }
}
```

Нас интересует инструкция `invokevirtual euw.d(Leqt;)Leqt;` так как она повторяется ровно 3 раза, значит вероятно именно она отвечает за добавление кнопок.

Тут и можно увидеть настоящий тип целевого метода в байткоде, класс имеет деобфусцированное имя `euw`, значит метод принадлежит классу `TitleScreen`, а не `Screen`, в свою очередь `d` это деобфусцированное имя метода `addDrawableChild`, а `Leqt;` это класс дженерика, eqt класс выглядит так :

```java
public interface eqt extends eqn {
   long B = 250L;

   default void e(double $$0, double $$1) {
   }

   default boolean a(double $$0, double $$1, int $$2) {
      return false;
   }

   default boolean b(double $$0, double $$1, int $$2) {
      return false;
   }

   default boolean a(double $$0, double $$1, int $$2, double $$3, double $$4) {
      return false;
   }

   default boolean a(double $$0, double $$1, double $$2) {
      return false;
   }

   default boolean a(int $$0, int $$1, int $$2) {
      return false;
   }

   default boolean b(int $$0, int $$1, int $$2) {
      return false;
   }

   default boolean a(char $$0, int $$1) {
      return false;
   }

   @Nullable
   default eou a(esv $$0) {
      return null;
   }

   default boolean a_(double $$0, double $$1) {
      return false;
   }

   void b_(boolean var1);

   boolean aB_();

   @Nullable
   default eou aF_() {
      return this.aB_() ? eou.a(this) : null;
   }

   default esz s() {
      return esz.a();
   }
}
```

Легко проследить по константе и родителю, что это интерфейс `Element` :

```java
package net.minecraft.client.gui;

import net.fabricmc.api.EnvType;
import net.fabricmc.api.Environment;
import net.minecraft.client.gui.navigation.GuiNavigation;
import net.minecraft.client.gui.navigation.GuiNavigationPath;
import net.minecraft.client.gui.navigation.Navigable;
import org.jetbrains.annotations.Nullable;

/**
 * Base GUI interface for handling callbacks related to
 * keyboard or mouse actions.
 * 
 * Mouse coordinate is bounded by the size of the window in
 * pixels.
 */
@Environment(EnvType.CLIENT)
public interface Element extends Navigable {
	long MAX_DOUBLE_CLICK_INTERVAL = 250L;
// ...
}
```

Который в свою очередь реализуется типом дженерика `T` в методе `addDrawableChild`.